In [ ]:
# !pip install openai spacy tqdm scikit-learn numpy

In [ ]:
# !python -m spacy download en_core_web_sm
# !python -m spacy download en_core_web_lg

In [ ]:
import os
import json
import re
import ast
# import torch
import spacy
import numpy as np
import time
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from typing import List, Dict, Tuple, Set
from sentence_transformers import SentenceTransformer

# --- Configuration ---
# It's recommended to set your API key as an environment variable for security.
# In your terminal, run: export OPENAI_API_KEY='your_key_here'
api_key = os.environ.get("")# <<< PASTE YOUR KEY HERE
if not api_key:
    # Fallback to a placeholder if the environment variable is not set.
    # WARNING: Replace this with your actual key. Do not share publicly.
    api_key = "" # <<< PASTE YOUR KEY HERE
    if "" in api_key: # <<< PASTE YOUR KEY HERE
        print("ERROR: OpenAI API key not found.")
        print("Please set the OPENAI_API_KEY environment variable or replace the placeholder in the script.")
        exit()

# Data paths
TRAIN_DATA_PATH = "train.txt"
TEST_DATA_PATH = "test.txt"

# --- Helper Functions ---

def read_line_examples_from_file(data_path: str) -> Tuple[List[str], List[list]]:
    """Reads data from file, each line is: sent####labels"""
    sents, labels = [], []
    with open(data_path, 'r', encoding='UTF-8') as fp:
        for line in fp:
            line = line.strip()
            if line != '' and '####' in line:
                words, tuples_str = line.split('####', 1)
                try:
                    tuples = ast.literal_eval(tuples_str)
                    sents.append(words)
                    labels.append(tuples)
                except (ValueError, SyntaxError):
                    print(f"Skipping malformed line: {line}")
    return sents, labels

def calculate_metrics(all_preds: List[list], all_golds: List[list]) -> Dict[str, float]:
    """
    Calculates precision, recall, and F1 score with normalization
    (case-insensitive, whitespace-stripped) to match the fine-tuning script's logic.
    """
    total_tp, total_fp, total_fn = 0, 0, 0

    for pred_quads, gold_quads in zip(all_preds, all_golds):
        # Normalize and convert lists of lists to sets of tuples for efficient comparison
        # This logic mirrors the normalize_quad and quads_list_to_set functions

        normalized_gt_set: Set[Tuple[str, ...]] = {
            tuple(str(x).strip().lower() for x in quad)
            for quad in gold_quads if isinstance(quad, (list, tuple)) and len(quad) == 4
        }

        normalized_pred_set: Set[Tuple[str, ...]] = {
            tuple(str(x).strip().lower() for x in quad)
            for quad in pred_quads if isinstance(quad, (list, tuple)) and len(quad) == 4
        }

        total_tp += len(normalized_gt_set.intersection(normalized_pred_set))
        total_fp += len(normalized_pred_set - normalized_gt_set)
        total_fn += len(normalized_gt_set - normalized_pred_set)

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    scores = {
        'precision': precision * 100,
        'recall': recall * 100,
        'f1': f1_score * 100
    }
    return scores


class GPT4o_ASQPExtractor:
    """
    An extractor class designed to evaluate the GPT-4o model via the OpenAI API.
    """
    def __init__(self, api_key: str):
        print("Initializing OpenAI client for GPT-4o...")
        self.client = OpenAI(api_key=api_key)
        self.nlp_sm = spacy.load("en_core_web_sm")

        print("Client and tools initialized successfully!")

    def _create_messages(self, instruction: str, query: str, dependency_tree: str, few_shot_examples: List[Dict]) -> List[Dict]:
        """
        Formats the prompt into the official OpenAI messages format.
        """
        # The main instruction goes into the 'system' role.
        system_content = instruction

        few_shot_text = ""
        if few_shot_examples: # This block only runs if the list is not empty
            few_shot_text = "### Examples\nHere are some examples to guide you:\n"
            for example in few_shot_examples:
                few_shot_text += f"\nSentence: {example['sentence']}\nLabels: {str(example['labels'])}\n"
            few_shot_text += "\n### Now, based on instruction and these examples, perform the task for the following sentence:\n"

        # Assemble the final user content. The few_shot_text will be an empty string for the zero-shot case.
        user_content = f"{few_shot_text}Query: {query}\n"
        user_content += f"Dependency Tree: {dependency_tree}\n"
        user_content += "LABELS:"

        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content}
        ]
        return messages

    def _parse_output(self, output_text: str) -> List[list]:
        """
        Robustly parse the model's string output into a Python list of lists.

        Strategy:
        1) Try JSON parse (if the model returned a JSON array).
        2) Fallback to finding a top-level bracketed substring using bracket matching.
        3) ast.literal_eval() the extracted substring.
        4) Validate that the result is a list of 4-element lists/tuples.
        """
        # 1) Try JSON parse first (most reliable if model returns JSON)
        try:
            data = json.loads(output_text)
            if isinstance(data, list) and all(isinstance(i, (list, tuple)) and len(i) == 4 for i in data):
                return data
        except Exception:
            pass

        # 2) Find top-level bracketed substring using bracket matching
        start = output_text.find('[')
        if start == -1:
            return []

        stack = 0
        end = -1
        for i in range(start, len(output_text)):
            ch = output_text[i]
            if ch == '[':
                stack += 1
            elif ch == ']':
                stack -= 1
                if stack == 0:
                    end = i
                    break
        if end == -1:
            return []

        list_str = output_text[start:end+1]

        # 3) Try ast.literal_eval on the extracted substring
        try:
            parsed_list = ast.literal_eval(list_str)
            if isinstance(parsed_list, list) and all(isinstance(i, (list, tuple)) and len(i) == 4 for i in parsed_list):
                return parsed_list
        except Exception:
            return []

        return []


    def extract_quadruples(self, instruction: str, sentence: str, few_shot_examples: List[Dict]) -> List[list]:
        """
        Gets a prediction from the GPT-4o API for a given sentence.
        """
        doc = self.nlp_sm(sentence)
        dep_lines = [f"{token.text}-{token.head.text}->{token.dep_}" for token in doc]
        dependency_tree = "\n".join(dep_lines) # Join into a multi-line string

        messages = self._create_messages(instruction, sentence, dependency_tree, few_shot_examples)

        try:
            response = self.client.chat.completions.create(
                model="gpt-4o-mini", # or gpt-4o
                messages=messages,
                max_tokens=512,
                temperature=0.0 # Set to 0 for deterministic, factual output
            )
            response_text = response.choices[0].message.content
            parsed = self._parse_output(response_text)
            if not parsed:
                print("DEBUG: could not parse model output:")
                print(response_text)
        except Exception as e:
            print(f"  [API Error]: {e}. Retrying after 10 seconds...")
            time.sleep(10)
            try:
                response = self.client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=messages,
                    max_tokens=512,
                    temperature=0.0
                )
                response_text = response.choices[0].message.content
                parsed = self._parse_output(response_text)
                if not parsed:
                    print("DEBUG: could not parse model output (retry):")
                    print(response_text)
            except Exception as e_retry:
                print(f"  [API Retry Failed]: {e_retry}. Returning empty list.")
                return []

        return parsed


if __name__ == "__main__":
    # --- 1. Initialization ---
    NUM_FEW_SHOTS = 1

    extractor = GPT4o_ASQPExtractor(api_key=api_key)

    # --- 2. Data Loading and Few-Shot Retriever Setup ---
    print("\nLoading datasets and building the few-shot retriever...")
    train_sents, train_labels = read_line_examples_from_file(TRAIN_DATA_PATH)
    test_sents, test_labels = read_line_examples_from_file(TEST_DATA_PATH)

    nlp_lg = spacy.load("en_core_web_lg")
    print("Building few-shot retriever index (this may take a moment)...")
    train_embeddings = np.array([nlp_lg(sent).vector for sent in tqdm(train_sents)])

    def retrieve_few_shots(query_sentence: str, num_examples: int = 1) -> List[Dict]:
        if num_examples <= 0:
            return []
        query_embedding = nlp_lg(query_sentence).vector.reshape(1, -1)
        similarities = cosine_similarity(query_embedding, train_embeddings).flatten()
        top_indices = np.argsort(similarities)[-num_examples:][::-1]

        return [{"sentence": train_sents[idx], "labels": train_labels[idx]} for idx in top_indices]

    # --- 3. Define the Master Instruction ---
    instruction_prompt = """
ROLE:
You are an expert ASQP (aspect sentiment quadruple prediction) annotation bot. Your processing must be precise, systematic, and strictly follow all rules.

TASK DEFINITION:
Your sole task is to extract all sentiment quadruples from a user-provided sentence.
Your response MUST be a valid Python list of lists, with no other text or explanations.
The output format for each quadruple is a list of four strings:
[<Aspect Term>, <Aspect Category>, <Sentiment Polarity>, <Opinion Term>]

ELEMENT DEFINITIONS:
1.  Aspect Term: The specific noun or phrase being reviewed.
    - It MUST be a term from the restaurant domain (e.g., "food", "service", "waiter").
    - Use the exact string "NULL" if the opinion is general or the aspect is implicit.
2.  Aspect Category: The general class for the Aspect Term.
    - You MUST choose from this predefined list only:
        - `ambience general`
        - `drinks prices`
        - `drinks quality`
        - `drinks style_options`
        - `food prices`
        - `food quality`
        - `food style_options`
        - `location general`
        - `restaurant general`
        - `restaurant miscellaneous`
        - `restaurant prices`
        - `service general`
3.  Sentiment: The polarity of the opinion.
    - It MUST be one of three strings: "positive", "negative", or "neutral".
4.  Opinion Term: The exact word(s) from the sentence expressing the sentiment.

EXTRACTION RULES:
1.  Concise Spans: The Opinion Term must be the most concise phrase that still captures the full sentiment. Do not include unnecessary words.
2.  Extract All Opinions: You must extract a separate quadruple for every distinct opinion expressed in the sentence.
3.  Implied Sentiment: Analyze comparative and conditional phrases to determine the implied sentiment.

STEP BY STEP PROCESS:
To ensure accuracy, think step-by-step before generating the final list:
1.  First, identify all opinion-expressing words/phrases in the sentence.
2.  Second, for each opinion, identify its specific aspect phrase. If none exist, that aspect is "NULL".
3.  Third, determine the sentiment polarity (positive, negative, neutral) and the correct aspect category from the provided list.
4.  Finally, construct the list of all quadruples according to all rules above.

FINAL OUTPUT INSTRUCTION:
Your final output MUST ONLY be a valid Python list of lists.
- Do not include explanations, reasoning, or markdown formatting.
- Do not include extra text before or after the list.
- Each inner list must contain exactly 4 strings.
"""

    # --- 4. Run Evaluation ---
    all_predictions = []
    all_ground_truths = test_labels

    print(f"\n--- Starting Evaluation on {len(test_sents)} Test Samples ---")
    for i, sentence in enumerate(tqdm(test_sents)):
        few_shot_examples = retrieve_few_shots(sentence, num_examples=NUM_FEW_SHOTS)
        predicted_quads = extractor.extract_quadruples(instruction_prompt, sentence, few_shot_examples)
        all_predictions.append(predicted_quads)

        print(f"\n----- Sample {i+1}/{len(test_sents)} -----")
        print(f"  Sentence:     {sentence}")
        print(f"  Ground Truth: {all_ground_truths[i]}")
        print(f"  Prediction:   {predicted_quads}")
        print("-" * 25)

        # Add a delay to avoid hitting API rate limits
        time.sleep(2)

    # --- 5. Report Final Metrics ---
    final_metrics = calculate_metrics(all_predictions, all_ground_truths)

    print("\n\n========== EVALUATION COMPLETE ==========")
    print(f"  Precision: {final_metrics['precision']:.2f}%")
    print(f"  Recall:    {final_metrics['recall']:.2f}%")
    print(f"  F1-Score:  {final_metrics['f1']:.2f}%")
    print("=======================================")

    # Optional: Save results to a file for detailed analysis
    results_to_save = []
    for i in range(len(test_sents)):
        results_to_save.append({
            "sentence": test_sents[i],
            "ground_truth": all_ground_truths[i],
            "prediction": all_predictions[i]
        })

    with open("Rest15_wo_DT_evaluation_1shot_results_gpt4o-mini.json", "w", encoding="utf-8") as f:
        json.dump(results_to_save, f, indent=4)
    print("\nDetailed results saved to 'Rest15_wo_DT_evaluation_1shot_results_gpt4o-mini.json'")

In [ ]:
import json
import unicodedata
import re
from typing import List, Dict, Tuple, Optional
from difflib import SequenceMatcher

# --- Helper Functions ---

def _normalize_text(s: str) -> str:
    """
    Normalizes unicode, lowercases, strips, removes punctuation,
    collapses whitespace, and removes leading/trailing prepositions/articles.
    """
    if not isinstance(s, str):
        s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.strip().lower()
    # Remove punctuation characters
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    # Define a list of common articles and prepositions to remove
    words_to_strip = [
        'a', 'an', 'the', 'of', 'in', 'on', 'at', 'for', 'to', 'with', 'by'
    ]

    # Split the string into words
    words = s.split()

    # Remove matching words from the beginning
    while words and words[0] in words_to_strip:
        words.pop(0)

    # Remove matching words from the end
    while words and words[-1] in words_to_strip:
        words.pop(-1)

    # Rejoin the words into a final string
    s = " ".join(words)

    return s

def _opinion_sim(a: str, b: str) -> float:
    """
    Uses SequenceMatcher ratio as a lightweight similarity metric for strings.
    """
    return SequenceMatcher(None, a, b).ratio()

# --- Advanced Metrics Calculation Function ---

def calculate_metrics(
    all_preds: List[List[List[str]]],
    all_golds: List[List[List[str]]],
    normalize: bool = True,
    dedupe: bool = True,
    opinion_fuzzy_threshold: Optional[float] = None
) -> Dict[str, float]:
    """
    Computes micro precision/recall/F1 for quadruple lists with advanced matching options.
    A quadruple is [Aspect Term, Aspect Category, Sentiment Polarity, Opinion Term].
    """
    if len(all_preds) != len(all_golds):
        raise ValueError(f"Length mismatch: preds {len(all_preds)} vs golds {len(all_golds)}")

    total_tp = total_fp = total_fn = 0

    for preds, golds in zip(all_preds, all_golds):
        # Prepare lists by normalizing text if specified
        # Quadruple format: [0:Aspect, 1:Category, 2:Sentiment, 3:Opinion]
        if normalize:
            # The new normalization rule is applied here to each element
            g_list = [tuple(_normalize_text(x) for x in label) for label in golds]
            p_list = [tuple(_normalize_text(x) for x in label) for label in preds]
        else:
            g_list = [tuple(str(x) for x in label) for label in golds]
            p_list = [tuple(str(x) for x in label) for label in preds]

        # Collapse duplicate quadruples within the same example if specified
        if dedupe:
            g_list = list(dict.fromkeys(g_list))
            p_list = list(dict.fromkeys(p_list))

        matched_pred_idx = set()
        tp = 0

        # Perform a greedy one-to-one matching
        for g in g_list:
            for pj, p in enumerate(p_list):
                if pj in matched_pred_idx:
                    continue

                # CORRECTED LOGIC:
                # Aspect (0), Category (1), and Sentiment (2) must match exactly.
                non_opinion_match = (g[0] == p[0] and g[1] == p[1] and g[2] == p[2])
                if not non_opinion_match:
                    continue

                # Check for opinion match (index 3), either exact or fuzzy
                opinion_match = False
                if opinion_fuzzy_threshold is None:
                    if g[3] == p[3]: # Exact match
                        opinion_match = True
                else:
                    sim = _opinion_sim(g[3], p[3]) # Fuzzy match
                    if sim >= opinion_fuzzy_threshold:
                        opinion_match = True

                if opinion_match:
                    tp += 1
                    matched_pred_idx.add(pj)
                    break # Move to the next gold quadruple

        fp = len(p_list) - tp
        fn = len(g_list) - tp

        total_tp += tp
        total_fp += fp
        total_fn += fn

    # Calculate final metrics
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Total True Positives: {total_tp}, False Positives: {total_fp}, False Negatives: {total_fn}")

    scores = {
        'precision': precision * 100,
        'recall': recall * 100,
        'f1': f1 * 100
    }
    return scores

def main():
    """
    Main function to load results from a JSON file and compute the F1 score.
    """
    # IMPORTANT: Update this path to point to your results file
    file_path = "/content/Rest15_wo_DT_evaluation_1shot_results_gpt4o-mini.json"
    all_predictions = []
    all_ground_truths = []

    print(f"Loading results from '{file_path}'...")

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            results_data = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found. Please check the path.")
        return
    except json.JSONDecodeError:
        print(f"Error: The file '{file_path}' is not a valid JSON file.")
        return

    # Extract the predictions and ground truth labels from the loaded data
    for item in results_data:
        all_predictions.append(item.get("prediction", []))
        all_ground_truths.append(item.get("ground_truth", []))

    if not all_predictions or not all_ground_truths:
        print("Error: Could not find 'prediction' or 'ground_truth' keys in the JSON file.")
        return

    print(f"Successfully loaded {len(all_predictions)} samples.")

    # --- Calculate and Report Final Metrics ---
    print("\nCalculating scores with advanced metrics...")

    # You can experiment with fuzzy matching by setting a threshold, e.g., opinion_fuzzy_threshold=0.85
    final_metrics = calculate_metrics(
        all_preds=all_predictions,
        all_golds=all_ground_truths,
        normalize=True,
        dedupe=True,
        opinion_fuzzy_threshold=None # Set to a float like 0.85 for fuzzy matching
    )

    print("\n\n========== FINAL METRICS ==========")
    print(f"  Precision: {final_metrics['precision']:.2f}%")
    print(f"  Recall:    {final_metrics['recall']:.2f}%")
    print(f"  F1-Score:  {final_metrics['f1']:.2f}%")
    print("===================================")


if __name__ == "__main__":
    main()